# Order Chatbot
We wanna automate the collection of user prompts and assistant responses to build a OrderBot. The OrderBot will take orders at a pizza restaurant.

In [7]:
import os
from openai import OpenAI
import dotenv
dotenv.load_dotenv()
from IPython.display import display, Markdown
import ipywidgets as widgets

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

In [8]:
panels = [] # store history

context = [ {'role':'system', 'content':"""
You are OrderBot, an automated service to collect orders for a pizza restaurant. 
You first greet the customer, then collects the order, 
and then asks if it's a pickup or delivery. 
You wait to collect the entire order, then summarize it and check for a final 
time if the customer wants to add anything else. 
If it's a delivery, you ask for an address. 
Finally you collect the payment.
Make sure to clarify all options, extras and sizes to uniquely 
identify the item from the menu.
You respond in a short, very conversational friendly style. 
The menu includes 
pepperoni pizza  12.95, 10.00, 7.00 
cheese pizza   10.95, 9.25, 6.50 
eggplant pizza   11.95, 9.75, 6.75 
fries 4.50, 3.50 
greek salad 7.25 
Toppings: 
extra cheese 2.00, 
mushrooms 1.50 
sausage 3.00 
canadian bacon 3.50 
AI sauce 1.50 
peppers 1.00 
Drinks: 
coke 3.00, 2.00, 1.00 
sprite 3.00, 2.00, 1.00 
bottled water 5.00 
"""} ]  # accumulate messages

In [9]:
def collect_messages(_):
    prompt = inp.value
    inp.value = ''
    context.append({'role':'user', 'content':f"{prompt}"})
    response = get_completion_from_messages(context) 
    context.append({'role':'assistant', 'content':f"{response}"})
    
    chat_history.children = list(chat_history.children) + [
        widgets.HTML(f"<b>User:</b> {prompt}"),
        widgets.HTML(f"<b>Assistant:</b> {response}")
    ]
    
    return response

In [13]:
def get_completion_from_messages(messages, temperature=0):
    response = client.chat.completions.create(
        model="gpt-4o",
        messages=messages,
        temperature=0,
    )
    return response.choices[0].message.content

In [11]:
# Chat interface using ipywidgets
chat_history = widgets.VBox([])
inp = widgets.Text(value="Hi", placeholder='Enter text here…', description='You:')
button = widgets.Button(description='Send')

def on_button_click(b):
    collect_messages(None)

button.on_click(on_button_click)

display(widgets.VBox([widgets.HBox([inp, button]), chat_history]))

In [14]:
messages =  context.copy()
messages.append(
{'role':'system', 'content':'create a json summary of the previous food order. Itemize the price for each item\
 The fields should be 1) pizza, include size 2) list of toppings 3) list of drinks, include size   4) list of sides include size  5)total price '},    
)   

response = get_completion_from_messages(messages, temperature=0)
print(response)

```json
{
  "pizza": {
    "type": "Cheese Pizza",
    "size": "Medium",
    "price": 9.25
  },
  "toppings": [
    {
      "type": "Extra Cheese",
      "price": 2.00
    },
    {
      "type": "Sausage",
      "price": 3.00
    }
  ],
  "drinks": [
    {
      "type": "Sprite",
      "size": "Medium",
      "price": 2.00
    }
  ],
  "sides": [],
  "total_price": 16.25
}
```
